In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import pickle
import torch

C:\Users\BPatel\anaconda3\envs\sindex\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('openalex_topics.csv')

# Combine text for the model to "read"
df['search_profile'] = (
    df['topic_name'].fillna('') + " " + 
    df['keywords'].fillna('') + " " + 
    df['summary'].fillna('')
)

# Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
topic_embeddings = model.encode(df['search_profile'].tolist(), show_progress_bar=True)

# Save everything to a single file
data_to_save = {
    'embeddings': topic_embeddings,
    'metadata': df
}

with open('openalex_topic_index.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)

print("Embedding file 'openalex_topic_index.pkl' created successfully.")

C:\Users\BPatel\anaconda3\envs\sindex\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BPatel\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|███████████████████████████████████████████████████████████████████████| 142/

Embedding file 'openalex_topic_index.pkl' created successfully.


In [7]:
def get_best_topic(metadata, index_path='openalex_topic_index.pkl'):
    # 1. Load the pre-calculated data
    with open(index_path, 'rb') as f:
        data = pickle.load(f)
    
    topic_embeddings = data['embeddings']
    df_topics = data['metadata']
    
    # 2. Initialize the SAME model used during setup
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # 3. Process the input metadata
    query_text = f"{metadata.get('title', '')} {' '.join(metadata.get('subjects', []))}"
    query_embedding = model.encode(query_text)
    
    # 4. Find the closest match
    cos_scores = util.cos_sim(query_embedding, topic_embeddings)[0]
    top_result_idx = torch.argmax(cos_scores).item()
    
    # 5. Get the actual confidence score
    confidence_score = cos_scores[top_result_idx].item()
    
    # Return both the metadata and the score
    result_row = df_topics.iloc[top_result_idx].to_dict()
    result_row['match_confidence'] = confidence_score
    
    # Return the best matching row
    return result_row

my_metadata = {
    "title": "Excavations at St Peter's Church, Barton-upon-Humber",
    "subjects": ["Archaeology", "Human Bone", "INHUMATION CEMETERY"]
}

best_match = get_best_topic(my_metadata, index_path='openalex_topic_index.pkl')

print(f"Topic: {best_match['topic_name']} (ID: {best_match['topic_id']})")
print(f"Confidence: {best_match['match_confidence']:.4f}")

Topic: Paleopathology and ancient diseases (ID: 13409)
Confidence: 0.4330


In [8]:
my_metadata = {"source": "emdb", "identifiers": [{"identifier": "EMD-48024", "identifier_type": "emdb_id"}], "url": "https://www.ebi.ac.uk/emdb/EMD-48024", "title": "map beta Paf1C", "subjects": ["Transcription", "SETD2", "H3K36me3"], "publication_date": "2024-11-21T00:00:00", "creators": [{"name": "Markert J", "name_type": "Personal"}, {"name": "Farnung L", "name_type": "Personal"}], "publisher": "The Electron Microscopy Data Bank (EMDB)"}
best_match = get_best_topic(my_metadata, index_path='openalex_topic_index.pkl')

print(f"Topic: {best_match['topic_name']} (ID: {best_match['topic_id']})")
print(f"Confidence: {best_match['match_confidence']:.4f}")

Topic: Melanoma and MAPK Pathways (ID: 11533)
Confidence: 0.4287


In [9]:
my_metadata = {"source": "datacite", "identifiers": [{"identifier": "10.13026/kpb9-mt58", "identifier_type": "doi"}], "doi": "10.13026/kpb9-mt58", "url": "https://physionet.org/content/mimiciv/3.1/", "title": "MIMIC-IV", "description": "Retrospectively collected medical data has the opportunity to improve patient\ncare through knowledge discovery and algorithm development. Broad reuse of\nmedical data is desirable for the greatest public good, but data sharing must\nbe done in a manner which protects patient privacy. Here we present Medical\nInformation Mart for Intensive Care (MIMIC)-IV, a large deidentified dataset\nof patients admitted to the emergency department or an intensive care unit at\nthe Beth Israel Deaconess Medical Center in Boston, MA. MIMIC-IV contains data\nfor over 65,000 patients admitted to an ICU and over 200,000 patients admitted\nto the emergency department. MIMIC-IV incorporates contemporary data and\nadopts a modular approach to data organization, highlighting data provenance\nand facilitating both individual and combined use of disparate data sources.\nMIMIC-IV is intended to carry on the success of MIMIC-III and support a broad\nset of applications within healthcare.", "version": "3.1", "publisher": "PhysioNet", "publication_date": "2024-10-10T21:27:19+00:00", "creators": [{"name": "Johnson, Alistair", "affiliations": ["Massachusetts Institute of Technology"]}, {"name": "Bulgarelli, Lucas", "affiliations": ["Massachusetts Institute of Technology"]}, {"name": "Pollard, Tom", "identifiers": ["https://orcid.org/0000-0002-5676-7898"], "affiliations": ["Massachusetts Institute of Technology, USA"]}, {"name": "Gow, Brian", "identifiers": ["https://orcid.org/0000-0002-7682-1943"], "affiliations": ["Massachusetts Institute of Technology"]}, {"name": "Moody, Benjamin", "affiliations": ["Massachusetts Institute of Technology"]}, {"name": "Horng, Steven", "affiliations": ["Beth Israel Deaconess Medical Center", "Harvard Medical School"]}, {"name": "Celi, Leo Anthony", "affiliations": ["Massachusetts Institute of Technology, Cambridge, USA", "Beth Israel Deaconess Medical Center, Boston, USA"]}, {"name": "Mark, Roger", "identifiers": ["https://orcid.org/0000-0002-6318-2978"], "affiliations": ["Massachusetts Institute of Technology"]}], "citations": {"dois": ["10.1093/jamia/ocae303", "10.1101/2025.02.11.25321833", "10.32628/cseit251112256", "10.2196/preprints.74142", "10.1101/2025.03.26.25324714", "10.1002/alz.14564", "10.1016/j.numecd.2025.103973", "10.1186/s40560-025-00797-9", "10.1089/pmr.2025.0015", "10.7759/cureus.86370", "10.1101/2025.06.26.25330281", "10.1186/s12871-025-03243-3", "10.1007/s40520-025-03115-3", "10.1038/s41598-025-14028-6", "10.21203/rs.3.rs-6680914/v1", "10.1038/s41598-025-12496-4", "10.1101/2025.07.20.25322556", "10.1186/s40779-025-00641-z", "10.1101/2025.08.15.25333725", "10.3389/fphar.2025.1622440", "10.2196/74142", "10.1145/3705328.3748022", "10.1530/ec-25-0377", "10.1093/gigascience/giaf107", "10.1038/s41597-025-05915-8", "10.1016/j.diabres.2025.112105", "10.1186/s13054-025-05642-x", "10.1016/j.jad.2025.120465", "10.1101/2025.08.25.25334266", "10.21203/rs.3.rs-7886784/v1", "10.3389/fcvm.2025.1599318", "10.1016/j.aucc.2025.101489", "10.1186/s12872-025-05231-4", "10.1101/2025.11.11.25340038"]}}
best_match = get_best_topic(my_metadata, index_path='openalex_topic_index.pkl')

print(f"Topic: {best_match['topic_name']} (ID: {best_match['topic_id']})")
print(f"Confidence: {best_match['match_confidence']:.4f}")

Topic: Human Motion and Animation (ID: 12290)
Confidence: 0.2369


In [10]:
my_metadata = {"source": "datacite", "identifiers": [{"identifier": "10.60775/fairhub.2", "identifier_type": "doi"}], "doi": "10.60775/fairhub.2", "url": "https://fairhub.io/datasets/2", "title": "Flagship Dataset of Type 2 Diabetes from the AI-READI Project", "subjects": ["Diabetes mellitus", "Machine Learning", "Artificial Intelligence", "Electrocardiography", "Continuous Glucose Monitoring"], "description": "This dataset contains data from 1067 participants that was collected between July 19, 2023 and July, 31 2024. Data from multiple modalities are included. The data in this dataset contain no protected health information (PHI). Information related to the sex and race/ethnicity of the participants as well as medication used has also been removed. A detailed description of the dataset is available in the AI-READI documentation for v2.0.0 of the dataset at https://docs.aireadi.org", "version": "2.0.0", "publisher": "FAIRhub", "publication_date": "2024-10-28T17:30:18+00:00", "creators": [{"name": "AI-READI Consortium", "name_type": "Organizational"}], "citations": {"dois": ["10.2196/preprints.83154", "10.2196/83154"]}}

best_match = get_best_topic(my_metadata, index_path='openalex_topic_index.pkl')

print(f"Topic: {best_match['topic_name']} (ID: {best_match['topic_id']})")
print(f"Confidence: {best_match['match_confidence']:.4f}")

Topic: Artificial Intelligence in Healthcare (ID: 11396)
Confidence: 0.5311
